# 18 — Debugging, Reproducibility, and PyTorch Best Practices

In the previous notebook, we studied regularization, initialization, and stable training.

Now we will focus on a skill that separates fragile experiments from reliable deep-learning work:

> **Systematic debugging and reproducible experimentation**

A PyTorch model can fail because of:

- Wrong shapes
- Wrong dtypes
- CPU/GPU mismatches
- Missing or unstable gradients
- `NaN` / `Inf`
- Data leakage
- Incorrect train/eval mode
- Uncontrolled randomness
- Broken preprocessing
- Incorrect checkpoint logic

## In this notebook, we will study:

- A systematic PyTorch debugging workflow
- Shape debugging
- Dtype debugging
- Device debugging
- Gradient debugging
- Detecting NaNs and infinities
- Reproducibility
- Random seeds
- Deterministic behavior
- Saving experiment configuration
- Model summaries
- Parameter inspection
- Data leakage checks
- Sanity-check experiments
- Overfitting one tiny batch
- Clean project organization
- Practical PyTorch best practices

## Main Goal

The debugging order to remember is:

$$
\boxed{
\text{Data}
\rightarrow
\text{Shape}
\rightarrow
\text{Dtype}
\rightarrow
\text{Device}
\rightarrow
\text{Forward}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradients}
\rightarrow
\text{Updates}
}
$$

The central idea is:

> **Find the first place where your assumptions stop being true.**


In [ ]:
import json
import math
import platform
import random
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader

print("PyTorch:", torch.__version__)
print("Python:", platform.python_version())


# 1. A Systematic Debugging Workflow

When training fails, do not immediately change the optimizer or architecture.

Check the pipeline in order:

$$
\boxed{
\begin{array}{c}
\text{Raw sample}\\
\downarrow\\
\text{Transformed sample}\\
\downarrow\\
\text{DataLoader batch}\\
\downarrow\\
\text{Model input}\\
\downarrow\\
\text{Model output}\\
\downarrow\\
\text{Loss}\\
\downarrow\\
\text{Gradients}\\
\downarrow\\
\text{Parameter update}
\end{array}
}
$$

At each point, verify:

- Shape
- Dtype
- Device
- Value range
- Finite values


# 2. The First Debugging Printout

Before a long training run, inspect:

- Input shape
- Target shape
- Input dtype
- Target dtype
- Input device
- Target device
- Model output shape
- Loss value


In [ ]:
torch.manual_seed(42)

inputs = torch.randn(8, 10)
targets = torch.randint(0, 3, (8,))

model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()

outputs = model(inputs)
loss = criterion(outputs, targets)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Input dtype:", inputs.dtype)
print("Target dtype:", targets.dtype)
print("Input device:", inputs.device)
print("Target device:", targets.device)
print("Output shape:", outputs.shape)
print("Loss:", loss.item())


# 3. Shape Debugging

Shape errors are among the most common PyTorch bugs.

Do not read:

$$
(32,\ 3,\ 224,\ 224)
$$

as four arbitrary numbers.

Interpret the dimensions:

$$
\begin{array}{|c|c|}
\hline
32 & \text{Batch size} \\
\hline
3 & \text{Channels} \\
\hline
224 & \text{Height} \\
\hline
224 & \text{Width} \\
\hline
\end{array}
$$

Every dimension should have a meaning.


# 4. Linear-Layer Shape Rule

For:

```python
nn.Linear(10, 5)
```

the last input dimension must be:

$$
10
$$

So:

$$
(32,\ 10)
\rightarrow
(32,\ 5)
$$


In [ ]:
layer = nn.Linear(10, 5)
x = torch.randn(32, 10)
y = layer(x)

print("Input:", x.shape)
print("Output:", y.shape)


# 5. Debugging a Linear-Layer Mismatch

If a layer expects 10 features but receives 12, the matrix multiplication cannot work.

A useful check is:

```python
print(x.shape[-1])
print(layer.in_features)
```


In [ ]:
layer = nn.Linear(10, 5)
wrong_x = torch.randn(32, 12)

print("Received features:", wrong_x.shape[-1])
print("Expected features:", layer.in_features)


# 6. CNN Shape Debugging

PyTorch image batches usually follow:

$$
\boxed{
N,\ C,\ H,\ W
}
$$

For:

```python
nn.Conv2d(
    in_channels=1,
    out_channels=16,
    kernel_size=3
)
```

the input channel dimension must be:

$$
1
$$


In [ ]:
conv = nn.Conv2d(
    1,
    16,
    kernel_size=3,
    padding=1
)

images = torch.randn(
    8,
    1,
    64,
    64
)

features = conv(images)

print("Input:", images.shape)
print("Output:", features.shape)


# 7. Channel-Order Bugs

Some libraries use:

$$
N,\ H,\ W,\ C
$$

instead of PyTorch's:

$$
N,\ C,\ H,\ W
$$

If the current tensor is truly NHWC, convert with:

```python
x = x.permute(0, 3, 1, 2)
```

Do not permute blindly. First identify the meaning of every dimension.


In [ ]:
nhwc = torch.randn(
    8,
    64,
    64,
    3
)

nchw = nhwc.permute(
    0,
    3,
    1,
    2
)

print("NHWC:", nhwc.shape)
print("NCHW:", nchw.shape)


# 8. Flattening Bugs

For CNN output:

$$
(N,\ C,\ H,\ W)
$$

the usual flattening is:

$$
(N,\ C\times H\times W)
$$

Use:

```python
torch.flatten(x, start_dim=1)
```

This preserves the batch dimension.


In [ ]:
x = torch.randn(
    4,
    8,
    16,
    16
)

correct = torch.flatten(
    x,
    start_dim=1
)

incorrect = torch.flatten(x)

print("Original:", x.shape)
print("Correct:", correct.shape)
print("Incorrect:", incorrect.shape)


# 9. Silent Broadcasting Bugs

Some shape mistakes do not crash.

Suppose:

$$
predictions.shape=(32,\ 1)
$$

and:

$$
targets.shape=(32)
$$

An element-wise subtraction broadcasts to:

$$
(32,\ 32)
$$

This is likely wrong, but the code still runs.


In [ ]:
predictions = torch.randn(32, 1)
targets = torch.randn(32)

difference = predictions - targets

print("Predictions:", predictions.shape)
print("Targets:", targets.shape)
print("Difference:", difference.shape)


# 10. Shape Assertions

Assertions can make incorrect assumptions fail early.

For regression where exact matching is expected:

```python
assert predictions.shape == targets.shape
```


In [ ]:
predictions = torch.randn(32, 1)
targets = torch.randn(32, 1)

assert predictions.shape == targets.shape

print("Shapes match.")


# 11. A Tensor-Inspection Helper


In [ ]:
def describe_tensor(name, tensor):
    print(name)
    print("  shape:", tuple(tensor.shape))
    print("  dtype:", tensor.dtype)
    print("  device:", tensor.device)
    print("  requires_grad:", tensor.requires_grad)
    print()

describe_tensor("inputs", inputs)
describe_tensor("outputs", outputs)


# 12. Dtype Debugging

The loss function determines the expected target dtype.

$$
\begin{array}{|c|c|}
\hline
\textbf{Loss} & \textbf{Typical Target Dtype} \\
\hline
MSELoss & torch.float32 \\
\hline
L1Loss & torch.float32 \\
\hline
BCEWithLogitsLoss & torch.float32 \\
\hline
CrossEntropyLoss & torch.long \\
\hline
\end{array}
$$


# 13. Correct `CrossEntropyLoss` Targets

For standard multi-class classification:

- Logits: floating point
- Targets: class indices
- Target dtype: `torch.long`


In [ ]:
logits = torch.randn(8, 3)

targets = torch.randint(
    0,
    3,
    (8,),
    dtype=torch.long
)

loss = nn.CrossEntropyLoss()(
    logits,
    targets
)

print("Target dtype:", targets.dtype)
print("Loss:", loss.item())


# 14. Correct `BCEWithLogitsLoss` Targets

For binary classification:

- Logits and targets usually have matching shapes
- Targets are usually floating point


In [ ]:
logits = torch.randn(8, 1)

targets = torch.randint(
    0,
    2,
    (8, 1)
).float()

loss = nn.BCEWithLogitsLoss()(
    logits,
    targets
)

print("Target dtype:", targets.dtype)
print("Loss:", loss.item())


# 15. Integer Image Inputs

Images may arrive as:

`torch.uint8`

with values:

$$
0,\ldots,255
$$

Most neural-network layers expect floating-point input.

Convert deliberately and apply the intended scaling.


In [ ]:
image_uint8 = torch.randint(
    0,
    256,
    (1, 64, 64),
    dtype=torch.uint8
)

image_float = (
    image_uint8.float()
    / 255.0
)

print("Before:", image_uint8.dtype)
print("After:", image_float.dtype)
print("Range:", image_float.min().item(), image_float.max().item())


# 16. Device Debugging

A model and its input tensors usually need to be on the same device.

Typical pattern:

```python
model = model.to(device)
inputs = inputs.to(device)
targets = targets.to(device)
```


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = nn.Linear(
    10,
    3
).to(device)

x = torch.randn(
    4,
    10
).to(device)

targets = torch.randint(
    0,
    3,
    (4,)
).to(device)

print("Model:", next(model.parameters()).device)
print("Input:", x.device)
print("Target:", targets.device)


# 17. Gradient Debugging

A valid forward pass does not guarantee learning.

Ask:

1. Do gradients exist?
2. Are they finite?
3. Are they extremely small?
4. Are they extremely large?
5. Do parameters change after `optimizer.step()`?


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

x = torch.randn(8, 10)
targets = torch.randint(0, 3, (8,))

criterion = nn.CrossEntropyLoss()

loss = criterion(
    model(x),
    targets
)

loss.backward()

for name, parameter in model.named_parameters():
    print(
        name,
        "| grad exists:",
        parameter.grad is not None
    )


# 18. Gradient Norms


In [ ]:
for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        print(
            name,
            "| grad norm:",
            parameter.grad.norm().item()
        )


# 19. Gradient-Finiteness Helper


In [ ]:
def check_gradients(model):
    all_finite = True

    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            print(name, "| grad = None")
            continue

        finite = torch.isfinite(
            parameter.grad
        ).all().item()

        norm = parameter.grad.norm().item()

        print(
            name,
            "| finite:",
            finite,
            "| norm:",
            norm
        )

        all_finite = all_finite and finite

    return all_finite

print(
    "All gradients finite:",
    check_gradients(model)
)


# 20. Why Can a Gradient Be `None`?

Possible causes:

- Parameter was not used in the forward pass
- `requires_grad=False`
- The tensor was detached
- The forward pass was inside `torch.no_grad()`
- The loss does not depend on that parameter

A `None` gradient is only a bug if you expected that parameter to participate.


# 21. Accidental `detach()`

`detach()` disconnects a tensor from the current Autograd graph.


In [ ]:
x = torch.tensor(
    2.0,
    requires_grad=True
)

y = x * 3
z = y.detach()

print("y requires_grad:", y.requires_grad)
print("z requires_grad:", z.requires_grad)


# 22. Checking Parameter Updates

Even when gradients exist, verify that parameters actually change.


In [ ]:
torch.manual_seed(42)

model = nn.Linear(10, 3)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

criterion = nn.CrossEntropyLoss()

x = torch.randn(8, 10)
targets = torch.randint(0, 3, (8,))

before = model.weight.detach().clone()

optimizer.zero_grad()

loss = criterion(
    model(x),
    targets
)

loss.backward()
optimizer.step()

after = model.weight.detach().clone()

print(
    "Weight changed:",
    not torch.equal(
        before,
        after
    )
)


# 23. Detecting NaNs and Infinities

Numerical problems may create:

- `nan`
- `inf`
- `-inf`

Use:

```python
torch.isfinite(...)
```


In [ ]:
values = torch.tensor([
    1.0,
    float("nan"),
    float("inf")
])

print(
    torch.isfinite(
        values
    )
)


# 24. A Finite-Tensor Assertion


In [ ]:
def assert_finite_tensor(name, tensor):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(
            f"{name} contains NaN or Inf."
        )

good_tensor = torch.randn(4, 4)

assert_finite_tensor(
    "good_tensor",
    good_tensor
)

print("Tensor is finite.")


# 25. Inspect Input Statistics

Before blaming the model, check preprocessing.

Useful quantities:

- Minimum
- Maximum
- Mean
- Standard deviation
- Finite status


In [ ]:
inputs = torch.randn(32, 10)

print("min:", inputs.min().item())
print("max:", inputs.max().item())
print("mean:", inputs.mean().item())
print("std:", inputs.std().item())
print(
    "finite:",
    torch.isfinite(inputs).all().item()
)


# 26. Check Output and Loss Finiteness


In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

outputs = model(inputs)

targets = torch.randint(
    0,
    3,
    (32,)
)

loss = nn.CrossEntropyLoss()(
    outputs,
    targets
)

print(
    "Output finite:",
    torch.isfinite(outputs).all().item()
)

print(
    "Loss finite:",
    torch.isfinite(loss).item()
)


# 27. Common Causes of `NaN` / `Inf`

Possible causes include:

- Learning rate too large
- Exploding gradients
- Invalid logarithms
- Division by zero
- Bad normalization
- Extremely large activations
- Mixed-precision overflow
- Corrupted input data
- Incorrect custom loss code

Find the **first** tensor that becomes non-finite.


# 28. Autograd Anomaly Detection

PyTorch provides:

```python
torch.autograd.detect_anomaly()
```

It can help identify problematic backward operations.

It is useful for debugging but can significantly slow execution.


In [ ]:
x = torch.tensor(
    2.0,
    requires_grad=True
)

with torch.autograd.detect_anomaly():
    y = x ** 2
    y.backward()

print("Gradient:", x.grad)


# 29. Reproducibility

Reproducibility means making experiments as repeatable as practical.

A reproducible experiment records enough information to recreate:

- Data split
- Model initialization
- Data order
- Hyperparameters
- Preprocessing
- Software environment
- Checkpoint selection

Exact bit-for-bit equality across every hardware/software combination is not always guaranteed.


# 30. Sources of Randomness

Randomness may come from:

- Python `random`
- PyTorch RNG
- CUDA RNG
- Weight initialization
- Data splitting
- DataLoader shuffling
- Data augmentation
- Some GPU kernels


# 31. A Seed Function


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(123)
a = torch.randn(5)

set_seed(123)
b = torch.randn(5)

print("A:", a)
print("B:", b)
print("Equal:", torch.equal(a, b))


# 32. Reproducible DataLoader Shuffling

Use a seeded `torch.Generator`.


In [ ]:
dataset = TensorDataset(
    torch.arange(20)
)

gen_a = torch.Generator().manual_seed(7)
gen_b = torch.Generator().manual_seed(7)

loader_a = DataLoader(
    dataset,
    batch_size=5,
    shuffle=True,
    generator=gen_a
)

loader_b = DataLoader(
    dataset,
    batch_size=5,
    shuffle=True,
    generator=gen_b
)

order_a = torch.cat([
    batch[0]
    for batch in loader_a
])

order_b = torch.cat([
    batch[0]
    for batch in loader_b
])

print("Same order:", torch.equal(order_a, order_b))


# 33. Deterministic Algorithms

PyTorch can request deterministic implementations:

```python
torch.use_deterministic_algorithms(True)
```

Tradeoffs:

- Some operations may not support deterministic execution
- Performance may decrease
- Some code may raise an error


In [ ]:
previous_setting = (
    torch.are_deterministic_algorithms_enabled()
)

torch.use_deterministic_algorithms(True)

print(
    "Enabled:",
    torch.are_deterministic_algorithms_enabled()
)

torch.use_deterministic_algorithms(
    previous_setting
)


# 34. CUDA / cuDNN Determinism

You may also encounter:

```python
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```

The exact reproducibility/performance tradeoff depends on:

- PyTorch version
- CUDA version
- cuDNN version
- GPU
- Operators used

Document the environment when reproducibility matters.


# 35. Reproducibility Is More Than a Seed

Even with fixed seeds, results can differ because of:

- Hardware
- Library versions
- Non-deterministic kernels
- Floating-point accumulation order
- DataLoader worker behavior

So save your experimental environment, not just the seed.


# 36. Saving Experiment Configuration

Important settings may include:

- Experiment name
- Seed
- Learning rate
- Batch size
- Weight decay
- Epochs
- Image size
- Model
- Loss
- Optimizer


In [ ]:
config = {
    "experiment_name": "cnn_baseline_v1",
    "seed": 42,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 20,
    "image_size": 64,
    "num_classes": 3,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss"
}

print(
    json.dumps(
        config,
        indent=2
    )
)


# 37. Saving Configuration to JSON


In [ ]:
config_path = Path(
    "experiment_config.json"
)

config_path.write_text(
    json.dumps(
        config,
        indent=2
    ),
    encoding="utf-8"
)

print("Saved:", config_path)


# 38. Saving Environment Information


In [ ]:
environment = {
    "python_version":
        platform.python_version(),

    "torch_version":
        torch.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_version":
        torch.version.cuda
}

if torch.cuda.is_available():
    environment["gpu_name"] = (
        torch.cuda.get_device_name(0)
    )

print(
    json.dumps(
        environment,
        indent=2
    )
)


# 39. Model Summaries

The simplest summary is:

```python
print(model)
```

It shows the module hierarchy.


In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

print(model)


# 40. Parameter Inspection

A useful parameter summary includes:

- Name
- Shape
- Number of scalar values
- Whether it is trainable


In [ ]:
def parameter_summary(model):
    total = 0
    trainable = 0

    for name, parameter in model.named_parameters():
        count = parameter.numel()
        total += count

        if parameter.requires_grad:
            trainable += count

        print(
            f"{name:25s} "
            f"shape={str(tuple(parameter.shape)):15s} "
            f"numel={count:6d} "
            f"trainable={parameter.requires_grad}"
        )

    print()
    print("Total:", total)
    print("Trainable:", trainable)

parameter_summary(model)


# 41. Inspecting Submodules


In [ ]:
for name, module in model.named_modules():
    print(
        name if name else "<root>",
        "->",
        module.__class__.__name__
    )


# 42. Forward Hooks for Shape Debugging

Forward hooks can print layer outputs automatically during one forward pass.


In [ ]:
def shape_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, torch.Tensor):
            print(
                f"{name:15s} -> "
                f"{tuple(output.shape)}"
            )

    return hook

hook_model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

handles = []

for name, module in hook_model.named_modules():
    if name:
        handles.append(
            module.register_forward_hook(
                shape_hook(name)
            )
        )

_ = hook_model(
    torch.randn(4, 10)
)

for handle in handles:
    handle.remove()


# 43. Why Remove Hooks?

Hooks stay attached until removed.

If you repeatedly register hooks without removing them:

- Output may appear multiple times
- Debugging becomes confusing
- Extra work happens every forward pass

Store the handles and remove them after inspection.


# 44. Data Leakage

Data leakage means information that should be unavailable during training influences:

- Model fitting
- Preprocessing
- Model selection
- Hyperparameter tuning
- Evaluation

Leakage can produce excellent-looking metrics that fail in real deployment.


# 45. Normalization Leakage

Wrong:

1. Compute mean/std using the full dataset
2. Split into train/validation/test

Better:

1. Split first
2. Compute statistics from training only
3. Apply training-derived statistics to validation/test


# 46. Patient-Level Leakage in Ultrasound

Suppose one patient contributes 20 ultrasound images.

If images are randomly split individually, the same patient can appear in both training and validation.

The model may learn:

- Patient-specific patterns
- Acquisition style
- Scanner settings

instead of the intended pathology.

When appropriate, split by:

- Patient
- Study
- Examination
- Acquisition session


# 47. Duplicate Leakage

Dangerous examples:

- Exact image copies
- Near-duplicate frames
- Crops from the same original image
- Augmented variants created before splitting

A safer strategy is:

> Split the original sample/unit first, then augment training data.


# 48. Label Leakage

Possible leakage sources include:

- File name contains the class
- Folder path contains diagnosis
- Metadata directly reveals outcome
- Image overlays correlate with the target
- Acquisition protocol is class-specific

Always inspect metadata as carefully as images.


# 49. Test-Set Leakage

The test set should not be repeatedly used to choose:

- Architecture
- Learning rate
- Dropout
- Threshold
- Number of epochs

Otherwise it becomes another validation set.


# 50. Data-Leakage Checklist

Ask:

1. Can the same patient appear in multiple splits?
2. Are duplicates present?
3. Were augmentations created before splitting?
4. Were normalization statistics fit on all data?
5. Does metadata reveal the label?
6. Do file names reveal class?
7. Is the test set being used for tuning?
8. Are site/device differences correlated with class?


# 51. Sanity-Check Experiments

Useful sanity checks include:

- One forward pass
- One backward pass
- Parameter-update check
- Overfit one tiny batch
- Random-label experiment
- Class-distribution inspection
- Disable augmentation
- Use `num_workers=0`
- Run on a tiny subset

These tests reduce a large problem into something inspectable.


# 52. Overfitting One Tiny Batch

One of the strongest debugging tests is:

> Can a sufficiently capable model memorize a very small fixed batch?

If not, suspect:

- Loss
- Targets
- Learning rate
- Model
- Gradients
- Optimizer
- Preprocessing


In [ ]:
set_seed(42)

tiny_x = torch.tensor([
    [-2.0, -2.0],
    [-2.0, -1.0],
    [2.0, 2.0],
    [2.0, 1.0],
    [-2.0, 2.0],
    [-1.0, 2.0]
])

tiny_y = torch.tensor([
    0,
    0,
    1,
    1,
    2,
    2
])

print(
    tiny_x.shape,
    tiny_y.shape
)


# 53. Tiny-Batch Model


In [ ]:
tiny_model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

tiny_criterion = nn.CrossEntropyLoss()

tiny_optimizer = torch.optim.Adam(
    tiny_model.parameters(),
    lr=0.05
)


# 54. Train on the Same Tiny Batch Repeatedly


In [ ]:
tiny_losses = []

for step in range(300):
    tiny_optimizer.zero_grad()

    logits = tiny_model(
        tiny_x
    )

    loss = tiny_criterion(
        logits,
        tiny_y
    )

    loss.backward()
    tiny_optimizer.step()

    tiny_losses.append(
        loss.item()
    )

with torch.no_grad():
    predictions = tiny_model(
        tiny_x
    ).argmax(
        dim=1
    )

    tiny_accuracy = (
        predictions
        == tiny_y
    ).float().mean()

print(
    "Final loss:",
    tiny_losses[-1]
)

print(
    "Tiny-batch accuracy:",
    tiny_accuracy.item()
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tiny_losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Overfitting One Tiny Batch")
plt.show()


# 55. If Tiny-Batch Overfitting Fails

Check:

- Output dimension
- Target values
- Target dtype
- Loss function
- Learning rate
- Frozen parameters
- Missing `backward()`
- Missing `step()`
- Accidental `detach()`
- Excessive regularization


# 56. Random-Label Sanity Check

A high-capacity model may memorize random **training** labels.

But without leakage, it should not generalize meaningfully to unseen random labels.

For balanced $C$-class classification, chance accuracy is roughly:

$$
\boxed{
\frac{1}{C}
}
$$

For 3 classes:

$$
\approx33.3\%
$$


# 57. Checking Class Distribution


In [ ]:
targets = torch.tensor([
    0, 0, 0, 0,
    1, 1,
    2
])

print(
    "Counts:",
    torch.bincount(
        targets
    )
)


# 58. Why Class Distribution Matters

If:

$$
95\%
$$

of samples belong to one class, a trivial classifier can achieve:

$$
95\%
$$

accuracy.

Always consider:

- Per-class metrics
- Confusion matrix
- Sensitivity / specificity
- F1
- AUROC / AUPRC

depending on the task.


# 59. Clean Project Organization

As projects grow, separate responsibilities.

A simple structure:

```text
project/
│
├── data/
├── notebooks/
├── src/
│   ├── datasets.py
│   ├── models.py
│   ├── train.py
│   ├── evaluate.py
│   └── utils.py
├── configs/
├── checkpoints/
├── outputs/
├── requirements.txt
└── README.md
```


# 60. What Goes Where?

## `datasets.py`

- Custom `Dataset`
- Data transforms
- Split logic
- DataLoader creation

## `models.py`

- `nn.Module` classes
- Model-building functions
- Custom layers

## `train.py`

- Training loop
- Validation loop
- Optimizer
- Scheduler
- Checkpointing

## `configs/`

- Hyperparameters
- Experiment settings


# 61. Best Practice — Use Clear Names

Prefer:

```python
train_images
val_targets
learning_rate
num_classes
```

over:

```python
a
b
x1
tmp
```

Clear naming makes debugging easier.


# 62. Best Practice — Keep Functions Small

Prefer small functions with one responsibility:

- `train_one_epoch()`
- `validate_one_epoch()`
- `save_checkpoint()`
- `collect_predictions()`

Small functions are easier to test.


# 63. Best Practice — Avoid Hidden Global State

Prefer explicit inputs:

```python
train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
)
```

rather than functions that depend on many global variables.


# 64. Best Practice — Assert Important Assumptions


In [ ]:
images = torch.randn(
    16,
    1,
    64,
    64
)

targets = torch.randint(
    0,
    3,
    (16,),
    dtype=torch.long
)

assert images.ndim == 4
assert images.shape[1] == 1
assert targets.dtype == torch.long
assert len(images) == len(targets)

print(
    "Assumptions verified."
)


# 65. Best Practice — Separate Training and Evaluation

Training:

- `model.train()`
- Gradients enabled
- `loss.backward()`
- `optimizer.step()`

Evaluation:

- `model.eval()`
- `torch.no_grad()`
- No parameter updates


# 66. Best Practice — Save the Best Model

The last epoch is not automatically the best.

Track validation performance and save the best checkpoint according to a predefined metric.


# 67. Best Practice — Save Optimizer State When Resuming

For resuming training, a checkpoint can include:

```python
{
    "epoch": ...,
    "model_state_dict": ...,
    "optimizer_state_dict": ...,
    "best_val_loss": ...
}
```


# 68. Best Practice — Log What Matters

Useful logs include:

- Epoch
- Train loss
- Validation loss
- Train metric
- Validation metric
- Learning rate
- Best checkpoint
- Seed
- Experiment name


In [ ]:
model = nn.Linear(4, 2)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print(
    "Current LR:",
    optimizer.param_groups[0]["lr"]
)


# 69. Best Practice — Use `state_dict()`

Standard model persistence:

```python
torch.save(
    model.state_dict(),
    path
)
```

and:

```python
model.load_state_dict(
    state_dict
)
```


# 70. Best Practice — Do Not Call `forward()` Directly

Define:

```python
forward()
```

but normally call:

```python
model(x)
```

so PyTorch's module machinery is preserved.


# 71. Best Practice — Use `torch.no_grad()` for Inference


In [ ]:
model = nn.Linear(4, 2)

model.eval()

x = torch.randn(8, 4)

with torch.no_grad():
    predictions = model(x)

print(
    "Requires grad:",
    predictions.requires_grad
)


# 72. Best Practice — Inspect One Batch Before Full Training

Before a long experiment, verify:

1. One raw sample
2. One transformed sample
3. One batch
4. Batch shape
5. Dtypes
6. Device
7. Value range
8. Model output
9. Loss
10. Gradients
11. Parameter update


# 73. Best Practice — Start With a Baseline

Before adding complexity, build a simple baseline that is:

- Correct
- Reproducible
- Easy to debug

Then improve one component at a time.


# 74. Best Practice — Keep the Test Set Sacred

Use validation data for development.

Use test data only after development decisions are complete.

Repeated test tuning destroys its role as an unbiased final estimate.


# 75. Best Practice — Inspect Failure Cases

After training, inspect:

- Misclassified samples
- High-confidence errors
- Difficult subgroups
- Device/site effects
- Class-specific failures

Aggregate metrics alone can hide important problems.


# 76. Best Practice — Repeat Important Experiments With Multiple Seeds

One run can be lucky or unlucky.

For important comparisons, consider several seeds and report:

- Mean
- Standard deviation
- Number of runs


In [ ]:
def run_with_seed(seed):
    set_seed(seed)

    model = nn.Linear(
        4,
        2
    )

    return model.weight[
        0,
        0
    ].item()

for seed in [
    1,
    2,
    3
]:
    print(
        seed,
        run_with_seed(seed)
    )


# 77. Best Practice — Keep Raw Data Immutable

Keep:

- Raw data unchanged
- Processed data separately
- Preprocessing code versioned

This makes experiments easier to reproduce.


# 78. Best Practice — Save Split Definitions

For scientific experiments, save exact sample IDs for:

- Train
- Validation
- Test

A random seed alone may not be sufficient if the dataset changes later.


# 79. Programmatic Split-Overlap Check


In [ ]:
train_ids = {
    "A",
    "B",
    "C"
}

val_ids = {
    "D",
    "E"
}

test_ids = {
    "F",
    "G"
}

assert train_ids.isdisjoint(
    val_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert val_ids.isdisjoint(
    test_ids
)

print(
    "No split overlap."
)


# 80. Best Practice — Save Checkpoint Metadata


In [ ]:
checkpoint_metadata = {
    "epoch": 12,
    "best_val_loss": 0.42,
    "config": config,
    "torch_version": torch.__version__
}

print(
    checkpoint_metadata.keys()
)


# 81. A Practical Debugging Checklist

## Data

- Can I inspect one raw sample?
- Is the label correct?
- Are splits correct?
- Is preprocessing appropriate?

## Tensor

- Shape?
- Dtype?
- Device?
- Finite values?

## Model

- Does forward pass run?
- Is output shape correct?
- Are parameter counts reasonable?

## Loss

- Correct loss?
- Correct target format?
- Logits vs probabilities correct?

## Gradients

- Do they exist?
- Are they finite?
- Are norms reasonable?

## Optimizer

- Do parameters change?
- Is learning rate reasonable?

## Generalization

- Can one tiny batch be memorized?
- Is leakage possible?
- Are validation metrics believable?


# 82. Compact Debugging Utility


In [ ]:
def debug_batch(
    model,
    inputs,
    targets,
    criterion
):
    print("=== INPUTS ===")
    describe_tensor(
        "inputs",
        inputs
    )

    print(
        "finite:",
        torch.isfinite(
            inputs
        ).all().item()
    )

    print(
        "min:",
        inputs.min().item()
    )

    print(
        "max:",
        inputs.max().item()
    )

    print()

    print("=== TARGETS ===")
    describe_tensor(
        "targets",
        targets
    )

    print(
        "target min:",
        targets.min().item()
    )

    print(
        "target max:",
        targets.max().item()
    )

    print()

    print("=== FORWARD ===")

    outputs = model(
        inputs
    )

    describe_tensor(
        "outputs",
        outputs
    )

    assert_finite_tensor(
        "outputs",
        outputs
    )

    loss = criterion(
        outputs,
        targets
    )

    print(
        "loss:",
        loss.item()
    )

    print(
        "loss finite:",
        torch.isfinite(
            loss
        ).item()
    )

    return loss


In [ ]:
model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

inputs = torch.randn(8, 10)

targets = torch.randint(
    0,
    3,
    (8,)
)

criterion = nn.CrossEntropyLoss()

_ = debug_batch(
    model,
    inputs,
    targets,
    criterion
)


# 83. Common Mistake — Debugging the Full System First

When debugging:

- Remove augmentation
- Set `num_workers=0`
- Use a tiny dataset
- Use a simple model
- Disable schedulers
- Disable mixed precision
- Overfit one batch

Reintroduce complexity gradually.


# 84. Common Mistake — Trusting High Accuracy Immediately

Suspiciously high scores may come from:

- Leakage
- Duplicates
- Severe imbalance
- Train/validation overlap
- Patient overlap
- Target information in metadata

Verify the experimental design before trusting the metric.


# 85. Common Mistake — Ignoring Warnings

Warnings can reveal:

- Broadcasting
- Deprecated behavior
- Dtype conversions
- DataLoader problems

Do not suppress warnings before understanding them.


# 86. Common Mistake — Hiding Exceptions

Avoid:

```python
try:
    ...
except:
    pass
```

during development.

It hides the actual error.


# 87. Common Mistake — Dirty Notebook State

A notebook may work only because an old variable still exists.

A strong validation step is:

> Restart runtime and run all cells from the top.

A good tutorial notebook should be self-contained.


# 88. Common Mistake — Saving Weights Without Experiment Context

Weights are difficult to interpret if you do not know:

- Architecture
- Preprocessing
- Class mapping
- Image size
- Seed
- Hyperparameters

Save the experiment context with the model.


# 89. Common Mistake — Comparing Different Splits

Two models trained on different splits are not a clean comparison.

For controlled experiments, keep:

- Split
- Preprocessing
- Evaluation metric

fixed while changing the intended component.


# 90. Reproducibility Does Not Guarantee Validity

Perfectly reproducible code can reproduce a flawed experiment.

Reproducibility does not guarantee:

- Correct labels
- No leakage
- Proper scientific question
- Appropriate evaluation
- External generalization

It is necessary, but not sufficient.


# 91. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Write a tensor inspection helper that prints:

- Shape
- Dtype
- Device
- Min
- Max
- Finite status

## Exercise 2

For `nn.Linear(5,3)`, what input feature dimension is required?

## Exercise 3

Create a correct `CrossEntropyLoss` example.

## Exercise 4

Create a correct `BCEWithLogitsLoss` example.

## Exercise 5

Write a function that checks whether all existing gradients are finite.

## Exercise 6

Verify that one optimizer step changes a parameter.

## Exercise 7

Write a seed function for Python and PyTorch.

## Exercise 8

Save an experiment config to JSON.

## Exercise 9

Overfit one tiny batch.

## Exercise 10

Check that train, validation, and test IDs are pairwise disjoint.


# 92. Conceptual Challenges

## Challenge 1

Why can silent broadcasting be more dangerous than a runtime error?

## Challenge 2

Why does `CrossEntropyLoss` commonly require `torch.long` targets?

## Challenge 3

Why might a parameter's gradient be `None`?

## Challenge 4

Why should model and input devices match?

## Challenge 5

Why is tiny-batch overfitting useful?

## Challenge 6

Why does setting a seed not guarantee identical results across every system?

## Challenge 7

Why should normalization statistics come only from training data?

## Challenge 8

Why is patient-level splitting important in many ultrasound datasets?

## Challenge 9

Why should exact split definitions be saved?

## Challenge 10

Why is a high validation score not automatically trustworthy?


# 93. Exercise Solutions


In [ ]:
# Exercise 1
def inspect_tensor(name, tensor):
    print(name)
    print("shape:", tuple(tensor.shape))
    print("dtype:", tensor.dtype)
    print("device:", tensor.device)

    if tensor.numel() > 0:
        print("min:", tensor.min().item())
        print("max:", tensor.max().item())

    if tensor.is_floating_point():
        print(
            "finite:",
            torch.isfinite(
                tensor
            ).all().item()
        )

# Exercise 3
logits_ex3 = torch.randn(
    8,
    4
)

targets_ex3 = torch.randint(
    0,
    4,
    (8,),
    dtype=torch.long
)

loss_ex3 = nn.CrossEntropyLoss()(
    logits_ex3,
    targets_ex3
)

print(
    "Exercise 3:",
    loss_ex3.item()
)

# Exercise 4
logits_ex4 = torch.randn(
    8,
    1
)

targets_ex4 = torch.randint(
    0,
    2,
    (8, 1)
).float()

loss_ex4 = nn.BCEWithLogitsLoss()(
    logits_ex4,
    targets_ex4
)

print(
    "Exercise 4:",
    loss_ex4.item()
)


In [ ]:
# Exercise 5
def all_gradients_finite(model):
    for parameter in model.parameters():
        if parameter.grad is not None:
            if not torch.isfinite(
                parameter.grad
            ).all():
                return False

    return True

# Exercise 6
model_ex6 = nn.Linear(
    5,
    3
)

optimizer_ex6 = torch.optim.SGD(
    model_ex6.parameters(),
    lr=0.1
)

x_ex6 = torch.randn(
    8,
    5
)

y_ex6 = torch.randint(
    0,
    3,
    (8,)
)

before = (
    model_ex6.weight
    .detach()
    .clone()
)

optimizer_ex6.zero_grad()

loss = nn.CrossEntropyLoss()(
    model_ex6(
        x_ex6
    ),
    y_ex6
)

loss.backward()
optimizer_ex6.step()

after = (
    model_ex6.weight
    .detach()
    .clone()
)

print(
    "Exercise 6 changed:",
    not torch.equal(
        before,
        after
    )
)


In [ ]:
# Exercise 7
def exercise_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

exercise_seed(42)

# Exercise 8
exercise_config = {
    "seed": 42,
    "learning_rate": 1e-3,
    "batch_size": 32
}

Path(
    "exercise_config.json"
).write_text(
    json.dumps(
        exercise_config,
        indent=2
    ),
    encoding="utf-8"
)

# Exercise 10
train_ids_ex = {1, 2, 3}
val_ids_ex = {4, 5}
test_ids_ex = {6, 7}

assert train_ids_ex.isdisjoint(
    val_ids_ex
)

assert train_ids_ex.isdisjoint(
    test_ids_ex
)

assert val_ids_ex.isdisjoint(
    test_ids_ex
)

print(
    "No overlap."
)


# 94. Key Takeaways

In this notebook, we learned:

- A systematic PyTorch debugging workflow
- Shape debugging
- CNN channel-order debugging
- Flatten debugging
- Silent broadcasting
- Dtype debugging
- Device debugging
- Gradient debugging
- Gradient norms
- Missing gradients
- Parameter-update checks
- Detecting `NaN` and `Inf`
- Autograd anomaly detection
- Random seeds
- Reproducible DataLoader shuffling
- Deterministic algorithms
- Saving experiment configuration
- Saving environment information
- Model summaries
- Parameter inspection
- Forward hooks
- Data leakage
- Duplicate leakage
- Patient-level leakage
- Test-set leakage
- Sanity-check experiments
- Overfitting one tiny batch
- Class-distribution checks
- Clean project organization
- Practical PyTorch best practices

The debugging order to remember is:

$$
\boxed{
\text{Data}
\rightarrow
\text{Shape}
\rightarrow
\text{Dtype}
\rightarrow
\text{Device}
\rightarrow
\text{Forward}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient}
\rightarrow
\text{Update}
}
$$

A reliable result requires:

$$
\boxed{
\text{Correct Pipeline}
+
\text{No Leakage}
+
\text{Reproducible Setup}
+
\text{Appropriate Evaluation}
}
$$


# 95. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What should you inspect first when a training pipeline fails?
2. Why should every tensor dimension have a semantic meaning?
3. What is silent broadcasting?
4. What target dtype does `CrossEntropyLoss` commonly expect?
5. What target dtype does `BCEWithLogitsLoss` commonly expect?
6. Why must model and input devices usually match?
7. How do you check whether a gradient exists?
8. How do you check gradient finiteness?
9. Why verify parameter changes explicitly?
10. What can cause `NaN` / `Inf`?
11. What does anomaly detection help debug?
12. What does a random seed control?
13. Why does a seed not guarantee perfect reproducibility everywhere?
14. Why save experiment configuration?
15. What should a parameter summary contain?
16. What are forward hooks useful for?
17. What is data leakage?
18. Why should normalization use training-only statistics?
19. Why should augmentations be created after splitting?
20. Why can patient-level splitting matter in medical imaging?
21. Why is overfitting a tiny batch a useful test?
22. Why should class distribution be inspected?
23. Why save exact split definitions?
24. Why repeat important experiments using multiple seeds?
25. Why is reproducibility necessary but not sufficient for scientific validity?


# Next Notebook

# 19 — Transfer Learning and Pretrained Models

In the next notebook, we will study:

- What is transfer learning?
- Why pretrained models help
- Feature extraction
- Fine-tuning
- Freezing layers
- Replacing classifier heads
- Pretrained CNNs in `torchvision`
- Input normalization for pretrained models
- Adapting RGB models to grayscale images
- Training only the classifier
- Unfreezing deeper layers
- Different learning rates for different parameter groups
- Saving fine-tuned models
- Transfer learning for ultrasound images
- Common transfer-learning mistakes
